# Notebook 03 — Train the Prophet Forecaster

This notebook trains a per-signal Prophet model on cached coordinate data,
validates it on held-out years, and uploads artifacts to W&B.

In [ ]:
import sys; sys.path.insert(0, '..')
import json, pickle, redis, pandas as pd, matplotlib.pyplot as plt, wandb
from prophet import Prophet
from apps.api.config import settings

wandb.init(project='chronos-forecaster', name='nb03-exploration')
r = redis.from_url(settings.redis_url)

In [ ]:
# Load all cached histories
histories = []
for key in r.scan_iter('history:*'):
    raw = r.get(key)
    if raw:
        histories.append(json.loads(raw))
print(f'Loaded {len(histories)} histories')

In [ ]:
# Build training DataFrame for NDVI
records = []
for hist in histories:
    for pt in hist.get('ndvi', []):
        if pt['value'] is not None:
            records.append({'ds': pd.Timestamp(f"{pt['year']}-07-01"), 'y': pt['value']})

df_ndvi = pd.DataFrame(records).dropna()
print(f'NDVI training samples: {len(df_ndvi)}')
df_ndvi.describe()

In [ ]:
# Train-test split: hold out 2020–2025
train = df_ndvi[df_ndvi.ds.dt.year < 2020]
test  = df_ndvi[df_ndvi.ds.dt.year >= 2020]

model = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
model.fit(train)

future = model.make_future_dataframe(periods=10, freq='YE')
forecast = model.predict(future)
test_pred = forecast[forecast.ds.dt.year >= 2020].set_index(forecast.ds.dt.year)
test_actual = test.set_index(test.ds.dt.year)

mae = abs(test_pred['yhat'] - test_actual['y']).mean()
print(f'NDVI holdout MAE (2020–2025): {mae:.4f}')
wandb.log({'ndvi_holdout_mae': mae})

In [ ]:
fig = model.plot(forecast)
plt.title('NDVI Prophet forecast (all training locations)')
wandb.log({'ndvi_forecast_plot': wandb.Image(fig)})
plt.show()